# 32 — Multi-Tenant RAG

> **Tier 9 | Multi-Tenant / Production Isolation**

## The Problem

In production SaaS, multiple customers (tenants) share the same infrastructure.
Naïve approaches either:
- Use **one collection per tenant** → expensive (Qdrant Cloud charges per collection)
- Mix all tenants in one collection with **no isolation** → data leaks across tenants

## Solution: Payload-Based Tenant Isolation

Store all tenants in **one collection** but tag every vector with a `tenant_id` payload field.
Every query applies a mandatory `tenant_id` filter — tenants can never see each other's data.

### Architecture

```
Collection: multi_tenant_rag_32
  ├── Point (tenant_id="acme",   doc_id="policy_v1",   page=1, ...)
  ├── Point (tenant_id="acme",   doc_id="policy_v1",   page=2, ...)
  ├── Point (tenant_id="globex", doc_id="handbook",    page=1, ...)
  └── Point (tenant_id="initech",doc_id="compliance",  page=3, ...)
```

### Isolation layers

| Layer | Mechanism |
|---|---|
| **Query isolation** | Mandatory `tenant_id` FieldCondition on every search |
| **Write isolation** | `tenant_id` stamped at ingest, no cross-tenant upserts |
| **Delete isolation** | Delete by `(tenant_id, doc_id)` filter — never touches other tenants |
| **Payload index** | `PayloadSchemaType.KEYWORD` on `tenant_id` for O(1) filtered ANN |


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INGEST ["📥  INGEST (any tenant)"]
        DOC(["📄 doc"]) --> EXT["Extract + Chunk"]
        EXT --> STAMP["Stamp tenant_id on every chunk"]
        STAMP --> EMB["Embed"]
        EMB --> UPSERT[("Qdrant\n(shared collection)")]
    end
    subgraph QUERY ["🔍  QUERY (tenant-isolated)"]
        Q(["❓ Question + tenant_id"]) --> QE["Embed question"]
        QE --> FILTER["Filter: must tenant_id = T"]
        FILTER --> ANN["ANN search"]
        ANN --> HITS["Top-K hits (T only)"]
        HITS --> LLM["Claude — answer"]
    end
    subgraph ADMIN ["🔧  ADMIN"]
        DEL["Delete tenant doc"] --> FILT2["Filter: tenant_id=T AND doc_id=D"]
        FILT2 --> SCROLL["Scroll point IDs"]
        SCROLL --> QDEL[("Batch delete")]
    end
    style INGEST fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style QUERY  fill:#dcfce7,stroke:#16a34a,color:#14532d
    style ADMIN  fill:#fef9c3,stroke:#ca8a04,color:#713f12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                        "boto3", "qdrant-client", "pymupdf", "python-dotenv"])
print("Packages ready.")


In [ ]:
import os, json, time, uuid, hashlib
from pathlib import Path
from typing import List, Dict, Optional, Set
from dotenv import load_dotenv

import boto3
import fitz
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, PayloadSchemaType,
    FilterSelector
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK — pymupdf", fitz.__version__)


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"
QDRANT_URL      = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY  = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME = "multi_tenant_rag_32"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 400
CHUNK_OVERLAP   = 40
TOP_K           = 4

# Three synthetic tenants, each with different domain documents
TENANTS = {
    "acme"   : {"domain": "healthcare analytics",   "color": "🔵"},
    "globex" : {"domain": "financial reporting",     "color": "🟢"},
    "initech": {"domain": "supply chain logistics",  "color": "🟡"},
}

print("Tenants:", list(TENANTS.keys()))
print(f"Collection: {COLLECTION_NAME}")


## Step 3 — Qdrant Setup (single shared collection)

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Cloud unavailable ({e}), falling back...')
    print('Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))

# Critical: payload indexes for O(1) filtered ANN
qdrant.create_payload_index(COLLECTION_NAME, "tenant_id", PayloadSchemaType.KEYWORD)
qdrant.create_payload_index(COLLECTION_NAME, "doc_id",    PayloadSchemaType.KEYWORD)
qdrant.create_payload_index(COLLECTION_NAME, "page",      PayloadSchemaType.INTEGER)

print(f'Collection "{COLLECTION_NAME}" ready with tenant_id index.')


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str]) -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        time.sleep(0.04)
    return out


def call_llm(system: str, user: str, max_tokens: int = 400) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]


print(f"Embed dim: {len(embed_text('multi-tenant rag test'))}")


## Step 5 — Tenant-Aware Ingest

Every chunk gets stamped with `tenant_id`. The chunk ID incorporates `tenant_id`
so the same document text can exist in multiple tenants without collision.


In [ ]:
def chunk_id(tenant_id: str, doc_id: str, page: int, text: str) -> str:
    key = f"{tenant_id}::{doc_id}::{page}::{text.strip()}"
    h = hashlib.sha256(key.encode('utf-8', errors='replace')).hexdigest()
    return str(uuid.UUID(h[:32]))


def recursive_split(text: str) -> List[str]:
    if len(text) <= CHUNK_SIZE:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        c = text[start:start + CHUNK_SIZE].strip()
        if c:
            chunks.append(c)
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def ingest_tenant_doc(tenant_id: str, doc_id: str,
                      pages: Dict[int, str]) -> Dict:
    """Ingest a dict of {page_num: text} under a specific tenant."""
    t0 = time.time()
    all_chunks = []
    for page, text in pages.items():
        for sub in recursive_split(text):
            cid = chunk_id(tenant_id, doc_id, page, sub)
            all_chunks.append({
                'id'       : cid,
                'text'     : sub,
                'tenant_id': tenant_id,
                'doc_id'   : doc_id,
                'page'     : page,
            })

    if not all_chunks:
        return {'tenant_id': tenant_id, 'doc_id': doc_id, 'chunks': 0}

    embs = embed_batch([c['text'] for c in all_chunks])
    pts  = [
        PointStruct(
            id     = all_chunks[i]['id'],
            vector = embs[i],
            payload= {
                'text'     : all_chunks[i]['text'],
                'tenant_id': all_chunks[i]['tenant_id'],
                'doc_id'   : all_chunks[i]['doc_id'],
                'page'     : all_chunks[i]['page'],
            }
        )
        for i in range(len(all_chunks))
    ]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
    elapsed = (time.time() - t0) * 1000
    print(f"  [{tenant_id}] doc={doc_id}  chunks={len(pts)}  {elapsed:.0f}ms")
    return {'tenant_id': tenant_id, 'doc_id': doc_id, 'chunks': len(pts), 'ms': elapsed}


print("ingest_tenant_doc() defined.")


## Step 6 — Tenant Document Data

Three tenants, each with two synthetic documents on their respective domains.


In [ ]:
TENANT_DOCS = {
    "acme": {
        "clinical_guidelines": {
            1: ("Acme Health Analytics Clinical Dashboard Guidelines. "
                "All clinical dashboards must display patient outcomes with 95% confidence intervals. "
                "Color coding: red for critical values above 120, yellow for borderline 80-120, green below 80. "
                "Mandatory fields: patient_id, admission_date, primary_diagnosis, attending_physician."),
            2: ("Acme Data Quality Standards. "
                "Missing values above 5% trigger automatic data quality alert. "
                "Lab results must be validated within 4 hours of entry. "
                "All duplicate records are merged using deterministic patient matching on DOB and SSN last 4 digits. "
                "Weekly data quality report sent to chief medical officer every Monday."),
            3: ("Acme Reporting SLA. "
                "Daily census reports generated at 06:00 EST. "
                "Monthly quality metrics due by 5th of each month. "
                "Emergency incident reports within 2 hours of event. "
                "All reports archived for 7 years per HIPAA retention requirements."),
        },
        "patient_privacy_policy": {
            1: ("Acme Patient Data Privacy Policy 2024. "
                "PHI access restricted to authorized clinical staff only. "
                "All PHI queries logged with user_id, timestamp, and data_element accessed. "
                "De-identification follows HIPAA Safe Harbor: 18 identifiers must be removed. "
                "Breach notification within 60 days to affected patients and HHS."),
            2: ("Acme Data Sharing Agreements. "
                "Research data sharing requires IRB approval and BAA with receiving institution. "
                "Minimum necessary standard applies: only data required for the stated purpose is shared. "
                "Third-party analytics vendors must pass annual SOC 2 Type II audit. "
                "Patient consent required for any secondary use of identifiable data."),
        },
    },
    "globex": {
        "financial_reporting_manual": {
            1: ("Globex Financial Reporting Manual Q4 2024. "
                "Revenue recognition per ASC 606: five-step model applies to all contracts. "
                "Material contracts above 5M USD require quarterly impairment testing. "
                "EBITDA bridge reconciliation mandatory in all board packages. "
                "FX translation: functional currency USD, presentation currency as per subsidiary."),
            2: ("Globex Close Calendar. "
                "Month-end close: T+3 business days. Quarter-end close: T+5. Year-end: T+10. "
                "Journal entry cutoff 17:00 ET on close date. "
                "Intercompany eliminations posted by controllership team only. "
                "Variance explanations for any line item exceeding 10% or 500K USD threshold."),
            3: ("Globex Chart of Accounts structure. "
                "Asset accounts 1000-1999, Liability 2000-2999, Equity 3000-3999. "
                "Revenue accounts 4000-4999, COGS 5000-5999, OpEx 6000-7999. "
                "All new GL accounts require CFO approval and Finance Systems ticket. "
                "Retired accounts marked inactive, not deleted, for audit trail preservation."),
        },
        "audit_procedures": {
            1: ("Globex Internal Audit Procedures 2024. "
                "Risk-based audit plan approved by Audit Committee annually. "
                "High risk areas: revenue recognition, vendor payments, payroll, IT access. "
                "Sampling methodology: statistical sampling at 95% confidence for populations above 1000. "
                "Audit findings rated Critical, High, Medium, Low per internal risk matrix."),
            2: ("Globex SOX Compliance Procedures. "
                "ITGC controls tested quarterly: access management, change management, backup/recovery. "
                "Key controls identified and mapped to financial statement assertions. "
                "Management testing 70% of key controls; external auditors test remaining 30%. "
                "Material weakness disclosure timeline: 4 business days after identification."),
        },
    },
    "initech": {
        "logistics_operations_manual": {
            1: ("Initech Supply Chain Operations Manual v3. "
                "Warehouse zones A-D by SKU velocity: A=fast (daily pick), D=slow (monthly). "
                "FIFO rotation enforced for perishables and products with shelf life under 90 days. "
                "Inbound receiving SLA: same-day for LTL, next-day for FTL. "
                "Cycle count schedule: A-zone weekly, B monthly, C quarterly, D annually."),
            2: ("Initech Carrier Performance Standards. "
                "On-time delivery target: 98.5% for ground, 99.2% for expedite. "
                "Carriers below 95% OTD for two consecutive months placed on corrective action plan. "
                "Damage claim threshold: above 0.5% of shipments triggers root cause analysis. "
                "Preferred carrier list reviewed quarterly by procurement and logistics directors."),
            3: ("Initech Inventory Accuracy Standards. "
                "Annual physical inventory within +/- 0.5% variance acceptable. "
                "Shrinkage above 1% triggers loss prevention investigation. "
                "Cycle count discrepancies above 100 units require supervisor sign-off. "
                "Inventory system: SAP WM module, real-time RF scanning mandatory."),
        },
        "vendor_management_policy": {
            1: ("Initech Vendor Management Policy 2024. "
                "New vendor onboarding: W-9, insurance certificate, diversity certification required. "
                "Strategic vendors (spend above 1M/year) reviewed semi-annually by supply chain director. "
                "Single-source vendors require VP approval and documented business justification. "
                "Vendor scorecards: quality 40%, delivery 35%, cost 15%, innovation 10%."),
            2: ("Initech Procurement Thresholds. "
                "Purchase under 5K: manager approval. 5K-50K: director approval. "
                "Above 50K: VP Finance and Supply Chain co-approval. "
                "Emergency purchases above 100K require CEO notification within 24 hours. "
                "Competitive bids required for all purchases above 25K."),
        },
    },
}

print("Tenant document corpus defined:")
for t, docs in TENANT_DOCS.items():
    n_pages = sum(len(p) for p in docs.values())
    print(f"  [{t}] {len(docs)} docs, {n_pages} pages total")


## Step 7 — Ingest All Tenants

In [ ]:
print("Ingesting all tenants...")
print()

ingest_stats = []
for tenant_id, docs in TENANT_DOCS.items():
    color = TENANTS[tenant_id]['color']
    domain = TENANTS[tenant_id]['domain']
    print(f"{color} Tenant: {tenant_id} ({domain})")
    for doc_id, pages in docs.items():
        s = ingest_tenant_doc(tenant_id, doc_id, pages)
        ingest_stats.append(s)
    print()

total_vectors = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"Total vectors in collection: {total_vectors}")
print(f"Total chunks ingested: {sum(s['chunks'] for s in ingest_stats)}")


## Step 8 — Tenant-Isolated RAG Query

The key guarantee: **no query can ever return results from another tenant**.
The `tenant_id` filter is baked into every search — not optional.


In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant for {domain}.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def tenant_rag_query(tenant_id: str, question: str,
                     doc_id_filter: Optional[str] = None) -> Dict:
    """Query strictly isolated to one tenant. Cross-tenant retrieval is impossible."""
    t0   = time.time()
    qvec = embed_text(question)
    t_emb = (time.time() - t0) * 1000

    # Mandatory tenant isolation filter
    must_clauses = [FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id))]
    if doc_id_filter:
        must_clauses.append(FieldCondition(key="doc_id", match=MatchValue(value=doc_id_filter)))

    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec, limit=TOP_K,
        with_payload=True,
        query_filter=Filter(must=must_clauses))

    hits    = [{'text': p.payload['text'], 'page': p.payload.get('page', '?'),
                'doc' : p.payload.get('doc_id', '?'), 'score': p.score}
               for p in resp.points]
    context = '\n\n'.join(
        f"[{i+1}] {h['doc']} p{h['page']} score={h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(hits))

    domain  = TENANTS[tenant_id]['domain']
    answer  = call_llm(
        SYSTEM_PROMPT.format(domain=domain),
        f"Context:\n{context}\n\nQuestion: {question}")
    t_total = (time.time() - t0) * 1000

    return {
        'tenant_id': tenant_id, 'question': question,
        'answer'   : answer,    'hits'     : hits,
        'embed_ms' : t_emb,     'total_ms' : t_total,
    }


print("tenant_rag_query() defined.")


## Step 9 — Demo: Each Tenant Asks Domain Questions

In [ ]:
TENANT_QUESTIONS = {
    "acme": [
        "What are the mandatory fields for clinical dashboards?",
        "How quickly must a data breach be reported to patients?",
    ],
    "globex": [
        "What is the month-end close timeline at Globex?",
        "What sampling methodology is used for populations above 1000?",
    ],
    "initech": [
        "What is the on-time delivery target for ground shipments?",
        "What spend threshold requires VP Finance co-approval?",
    ],
}

print("=" * 65)
print("TENANT RAG QUERIES")
print("=" * 65)

all_results = {}
for tenant_id, questions in TENANT_QUESTIONS.items():
    color  = TENANTS[tenant_id]['color']
    domain = TENANTS[tenant_id]['domain']
    print(f"\n{color} Tenant: {tenant_id} ({domain})")
    print("-" * 55)
    all_results[tenant_id] = []
    for q in questions:
        r = tenant_rag_query(tenant_id, q)
        safe_ans = r['answer'].encode('ascii', 'replace').decode()
        print(f"Q: {q}")
        print(f"   hits: {[(h['doc'], h['score']) for h in r['hits'][:2]]}")
        print(f"A: {safe_ans[:260]}")
        print()
        all_results[tenant_id].append(r)


## Step 10 — Isolation Verification

Critical test: tenant A **must not** be able to retrieve tenant B's data,
even when asking an identical question.


In [ ]:
print("=" * 65)
print("ISOLATION VERIFICATION")
print("=" * 65)
print()

# Ask the same generic question from each tenant — answers must differ
question = "What are the key compliance and reporting requirements?"

print(f"Question (same for all): '{question}'")
print()

isolation_results = {}
for tenant_id in TENANTS:
    r    = tenant_rag_query(tenant_id, question)
    docs = list({h['doc'] for h in r['hits']})
    isolation_results[tenant_id] = docs
    color = TENANTS[tenant_id]['color']
    print(f"{color} {tenant_id}: retrieved docs = {docs}")

print()
# Verify: no tenant sees another tenant's documents
all_passed = True
for t1 in TENANTS:
    for t2 in TENANTS:
        if t1 == t2:
            continue
        t2_docs = set(d for docs in TENANT_DOCS[t2] for d in [docs])
        retrieved_by_t1 = set(isolation_results[t1])
        leaked = retrieved_by_t1 & t2_docs
        if leaked:
            print(f"FAIL: {t1} can see {t2}'s docs: {leaked}")
            all_passed = False

if all_passed:
    print("PASS: All tenants isolated — no cross-tenant document leakage.")


## Step 11 — Admin Operations: Delete Tenant Document

When a tenant removes a document (e.g. policy update), all vectors for that
`(tenant_id, doc_id)` must be purged without touching other tenants.


In [ ]:
def delete_tenant_doc(tenant_id: str, doc_id: str) -> int:
    """Delete all vectors for a specific (tenant, document). Returns deleted count."""
    doc_filter = Filter(must=[
        FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id)),
        FieldCondition(key="doc_id",    match=MatchValue(value=doc_id)),
    ])

    # Scroll to collect all point IDs
    ids_to_delete = []
    next_page = None
    while True:
        scroll_result, next_page = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=doc_filter,
            limit=100,
            offset=next_page,
            with_payload=False,
            with_vectors=False,
        )
        ids_to_delete.extend([p.id for p in scroll_result])
        if next_page is None:
            break

    if ids_to_delete:
        qdrant.delete(collection_name=COLLECTION_NAME,
                      points_selector=ids_to_delete)
    return len(ids_to_delete)


def count_tenant_vectors(tenant_id: str) -> int:
    filt = Filter(must=[FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id))])
    result, _ = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=filt,
        limit=500,
        with_payload=False, with_vectors=False)
    return len(result)


# ── Demo: delete acme's patient_privacy_policy ────────────────────────────────
print("=" * 65)
print("ADMIN: Delete tenant document")
print("=" * 65)
print()

before = count_tenant_vectors("acme")
print(f"acme vectors before delete: {before}")

deleted = delete_tenant_doc("acme", "patient_privacy_policy")
print(f"Deleted {deleted} vectors for acme/patient_privacy_policy")

after = count_tenant_vectors("acme")
print(f"acme vectors after delete : {after}")

# Globex and initech must be unaffected
globex_count  = count_tenant_vectors("globex")
initech_count = count_tenant_vectors("initech")
print(f"globex vectors (unchanged): {globex_count}")
print(f"initech vectors (unchanged): {initech_count}")

print()
assert after == before - deleted
print("PASS: Only acme/patient_privacy_policy vectors removed.")
print("PASS: Other tenants unaffected.")


## Step 12 — Admin Operations: Purge Entire Tenant

When a tenant churns or is offboarded, all their vectors are removed.


In [ ]:
def purge_tenant(tenant_id: str) -> int:
    """Delete ALL vectors for a tenant. Used on tenant offboarding."""
    filt = Filter(must=[FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id))])

    ids_to_delete = []
    next_page = None
    while True:
        scroll_result, next_page = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=filt,
            limit=200,
            offset=next_page,
            with_payload=False, with_vectors=False)
        ids_to_delete.extend([p.id for p in scroll_result])
        if next_page is None:
            break

    if ids_to_delete:
        qdrant.delete(collection_name=COLLECTION_NAME,
                      points_selector=ids_to_delete)
    return len(ids_to_delete)


print("=" * 65)
print("ADMIN: Purge entire tenant (offboarding)")
print("=" * 65)
print()

before_total = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"Total vectors before purge: {before_total}")

initech_before = count_tenant_vectors("initech")
purged = purge_tenant("initech")
initech_after  = count_tenant_vectors("initech")

after_total = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"initech vectors purged    : {purged}")
print(f"Total vectors after purge : {after_total}")
print(f"initech remaining         : {initech_after}")

print()
assert initech_after == 0
assert after_total == before_total - purged
print("PASS: initech fully purged.")
print("PASS: Other tenants unaffected.")


## Step 13 — Stats Report

In [ ]:
print("=" * 65)
print("MULTI-TENANT RAG — Stats Report")
print("=" * 65)
print()

# Ingest summary
print("Ingest summary:")
for s in ingest_stats:
    print(f"  [{s['tenant_id']}] {s['doc_id']:<35} {s['chunks']:>4} chunks  {s.get('ms',0):.0f}ms")

print()
print("Query latency summary (2 queries per tenant):")
for tenant_id, results in all_results.items():
    if results:
        avg_ms = sum(r['total_ms'] for r in results) / len(results)
        color  = TENANTS[tenant_id]['color']
        print(f"  {color} {tenant_id:<10} avg={avg_ms:.0f}ms")

print()
print("Final collection state:")
print(f"  Total vectors            : {qdrant.get_collection(COLLECTION_NAME).points_count}")
for t in ['acme', 'globex', 'initech']:
    cnt = count_tenant_vectors(t)
    print(f"  {TENANTS[t]['color']} {t:<12}: {cnt} vectors")

print()
print("Key design decisions:")
print("  1. One shared collection — cost-efficient (no per-tenant overhead)")
print("  2. tenant_id payload index (KEYWORD) — O(1) filtered ANN")
print("  3. chunk_id = UUID(SHA-256(tenant+doc+page+text)) — no cross-tenant collision")
print("  4. Mandatory filter on every query — isolation enforced at code level")
print("  5. Scroll-then-delete pattern — safe bulk delete by filter")
print()
print("Notebook 32 — Multi-Tenant RAG complete.")


## Summary

### Single collection, full tenant isolation

```python
# Every query — tenant filter is MANDATORY, never optional
resp = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=question_vector,
    query_filter=Filter(must=[
        FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id))
    ])
)
```

### Chunk ID prevents cross-tenant collision
```python
chunk_id = UUID(SHA-256(tenant_id + "::" + doc_id + "::" + page + "::" + text))
```

### Admin operations

| Operation | Mechanism |
|---|---|
| Delete document | Scroll by `(tenant_id, doc_id)` → batch delete |
| Purge tenant | Scroll by `tenant_id` → batch delete |
| Count tenant vectors | Scroll with tenant filter |

### Isolation guarantees

| Guarantee | How |
|---|---|
| Query isolation | Mandatory `tenant_id` filter on every ANN search |
| Write isolation | `tenant_id` stamped at ingest — never writable by other tenant |
| Delete isolation | Delete filter always scoped to `tenant_id` |
| ID isolation | UUID includes `tenant_id` — same text in 2 tenants → different IDs |

> **Next: [33 — Federated RAG](33_Federated_RAG.ipynb)**
